In [ ]:
import re
import json
from pathlib import Path

##Task 1 File ingestion and regular expressions

Function `read_file()`uses the Path to read a file, it is written to handel utf-8 or ISO-8859-1/Latin-1 encoding. It returns a tuple containing the files content and the used encoding as strings.

In [ ]:
def read_file(input_file: Path) -> tuple[str, str]:
    """Read a file and return its content and the encoding used.
    
    Arguments:
        input_file: Path to the file to read.
        
    Returns:
        A tuple containing the file content as a string and the encoding used as a string."""
    
    try:
        # Try to read the file with UTF-8 encoding
        with input_file.open(encoding="utf-8") as file:
            file_text_content = file.read()
            encoding_used = "utf-8"

    except UnicodeDecodeError:
        # If UTF-8 fails, try ISO-8859-1/Latin-1 encoding
        with input_file.open(encoding="ISO-8859-1") as file:
            file_text_content = file.read()
            encoding_used = "ISO-8859-1"

    # Returns a tuple with file_text_content and encoding_used
    return file_text_content, encoding_used


The function `parse_log()` parses the log file contents and returns the valid transactions as a list of dictionaries. 
The functions uses regular expresions to filter out the valid transactions. The regex pattern groups the different elements together and names each group. The different groups are then organised as dictionary keys. The filterd out groups are: timestamp, register_id, item, price, currency and status.

In [ ]:
def parse_log(file_text_content: str) -> list[dict[str, str | float]]:
    """Parse log file content and and return valid transactions.

    A parser function analyses text and extracts structured information from it according to a defined format.

    Arguments:
        file_text_content: The complete text content of a log file.

    Returns:
        A list of dictionaries containing valid transactions.
        Each dictionary contains the timestamt, register ID, item,
        price, currency, and status. The price is stored as a float.
        Lines that do not match the expected log format are ignored.
    """

    # Regex pattern for valid transactions
    # () - groups result together
    # ?P<...> names the group
    # r"..." raw string
    pattern = (
    r"\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\] " 
    r"\[(?P<register_id>\w+)\] " 
    r"item=(?P<item>\S+) " 
    r"price=(?P<price>\d+\.\d+) "  
    r"(?P<currency>EUR|NOK|USD) " 
    r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)" 
) 
    # empty list to collect the valid lines in the logfiles
    valid_transactions = []

    # splits up the file so we can work with it line by line
    for line in file_text_content.splitlines():

        # check whether this line matches the pattern
        match = re.fullmatch(pattern, line)

        # if the line is valid, extract this field
        if match:
            transaction = match.groupdict()

            # convert prise to float
            transaction["price"] = float(transaction["price"])

            # add the transaction to the list
            valid_transactions.append(transaction)

    return valid_transactions

## Task 2 recursive utility functions

The code `find_log_files()` is used to find .log files in a directory. It takes in a path to the startingpoint of the directory. Then it recursively searches down the path until it finds the .log files. It returns a list of path objects representing the log files it finds. 

In [ ]:
def find_log_files(directory: Path) -> list[Path]:
    # directory: Path - funksjonen mottar en mappebane
    # -> list[Path] - funksjonen skal returnere en liste med filbaner - ikke filinnholdet
    """Find all log files in a directory and its subdirectories.
    
    The function searches recursively through the directory structure 
    and collects the paths of all files ending in .log.
    
    Arguments:
        directory: Path to the directory where the search begins.
        
    Returns:
        A list of Path objects representing the log files found.
    """

    all_log_files = []

    for item in directory.iterdir():

        if item.is_dir():
            # Search the subdirectory recursively and collect its log files

            all_log_files.extend(find_log_files(item))  # This calls the function again for subfolders 

        elif item.is_file():
            # check file ending - .suffix gives fileending including . 
            if item.suffix == ".log":
                all_log_files.append(item)

    return all_log_files

The function `find_active_registers()` searches nested dictionaries and lists to find active register ids. This function is used on a the store_config.json file, witch gives information about the configuration of some of the stores in the company. 

In [ ]:
def find_active_registers(data: dict | list) -> set[str]:
    """Find IDs fo all active registers in nested data.
    
    Arguments:
        data: nested dictionaries and lists to search.
        
    Returns:
        A set of IDs belonging to active registers"""

    # empty set for collecting active registers
    active_registers = set()

    if isinstance(data, dict):
    # isinstance(item, dict) asks if the item a dictionary
        # Check wheter this dictionary describes an active register
        if (
            "id" in data
            and data.get("active") is True
        ):
            active_registers.add(data["id"])

        # Search recursively through the dictionary values
        for value in data.values():
            # data.values() gives the values inside the dict
            # each value gets sendt back to the same function
            # if it is a dict og a list the function "searches" it
            # .update() adds all the found ids into the existing set
            active_registers.update(find_active_registers(value))

    elif isinstance(data, list):
    # isinstance(item, list) asks if the item a list
        # Search recursively through each element in the list
        for item in data:
            active_registers.update(find_active_registers(item))

    return active_registers